# CoordinX - Phase 7: Inference and Explainability

Notebook ini menjalankan Phase 7 secara bertahap dari artifact canonical Phase 6.

**Urutan cell:**
1. Setup dan reproducibility
2. Validasi model dan input canonical
3. Jalankan inference seluruh akun
4. Validasi prediction, uncertainty, dan metadata
5. Tinjau evidence dan prioritas human review
6. Validasi artifact dan simpan ringkasan notebook

Catatan: model saat ini adalah TF-IDF + temporal + graph fusion proxy, bukan IndoBERTweet/LSTM/GraphSAGE final.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'PRD.md').exists())
sys.path.insert(0, str(ROOT / 'src'))

from inference.phase7_inference import run_inference
from models.phase6_advanced import MODEL_FEATURES

print(f'Repository: {ROOT}')
print(f'Model features: {len(MODEL_FEATURES)}')
print('Notebook setup: OK')

Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Model features: 30
Notebook setup: OK


In [2]:
required_paths = [
    ROOT / 'models' / 'phase6_fusion_proxy_combined_canonical.pkl',
    ROOT / 'data' / 'features' / 'batch2_account_features_with_scores.csv',
    ROOT / 'data' / 'features' / 'batch3_account_features_with_scores.csv',
    ROOT / 'data' / 'features' / 'batch2_validation.json',
    ROOT / 'data' / 'features' / 'batch3_validation.json',
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f'Missing Phase 7 input: {missing}'

for batch in ('batch2', 'batch3'):
    validation = json.loads((ROOT / 'data' / 'features' / f'{batch}_validation.json').read_text())
    print(f"{batch}: posts={validation['post']['posts']}, accounts={validation['account_features']['rows']}, labels={validation['label_preservation']['canonical_counts']}")
print('Canonical input validation: OK')

batch2: posts=1514, accounts=1007, labels={'0': 27, '1': 23}
batch3: posts=1081, accounts=796, labels={'0': 21, '1': 29}
Canonical input validation: OK


In [3]:
result = run_inference(ROOT, experiment='combined')
readiness = result['readiness']
predictions = result['predictions'].copy()
print(json.dumps(readiness, indent=2, default=str))

{
  "generated_at_utc": "2026-11-10T07:32:08.698275+00:00",
  "model_path": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/models/phase6_fusion_proxy_combined_canonical.pkl",
  "model_version": "phase6_fusion_proxy_combined_canonical.pkl:2026-11-10T05:51:22.939445+00:00",
  "schema_version": "canonical-v1",
  "rows": 1803,
  "batches": {
    "batch2": 1007,
    "batch3": 796
  },
  "predicted_buzzer": 440,
  "review_required": 291,
  "priority_review": 328,
  "mean_coordination_score": 0.33384306133809866,
  "artifacts": {
    "combined_predictions": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/data/predictions/phase7_combined_predictions.csv",
    "predictions_report": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase7/phase7_predictions.csv",
    "priority_review": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase7/phase7_priority_review.csv",
    "score_distribution": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase

In [4]:
required_columns = {
    'account_id', 'username', 'dataset_id', 'issue_id', 'prediction',
    'coordination_score', 'confidence', 'uncertainty',
    'text_evidence', 'temporal_evidence', 'graph_evidence',
    'positive_model_evidence', 'negative_model_evidence', 'review_status'
}
assert required_columns.issubset(predictions.columns)
assert len(predictions) == 1803
assert not predictions.duplicated(['dataset_id', 'account_id']).any()
assert predictions['coordination_score'].between(0, 1).all()
assert predictions['confidence'].between(0, 1).all()
assert predictions['uncertainty'].between(0, 1).all()
assert predictions['human_label'].notna().sum() == 100
assert set(predictions['review_status']) <= {'REVIEW_REQUIRED', 'PRIORITY_REVIEW', 'LOW_PRIORITY'}

print('Prediction rows:', len(predictions))
print('Predicted Buzzer:', int((predictions['prediction'] == 'Buzzer').sum()))
print('Review status:')
print(predictions['review_status'].value_counts().to_string())
print('Prediction validation: OK')

Prediction rows: 1803
Predicted Buzzer: 440
Review status:
review_status
LOW_PRIORITY       1184
PRIORITY_REVIEW     328
REVIEW_REQUIRED     291
Prediction validation: OK


In [5]:
summary = (predictions.groupby(['dataset_id', 'prediction'])['account_id']
           .count().unstack(fill_value=0))
print('Prediction summary by batch:')
print(summary.to_string())

review_columns = [
    'dataset_id', 'account_id', 'username', 'prediction',
    'coordination_score', 'uncertainty', 'text_evidence',
    'temporal_evidence', 'graph_evidence', 'review_status'
]
priority_review = (predictions[predictions['review_status'].isin(['REVIEW_REQUIRED', 'PRIORITY_REVIEW'])]
                   .sort_values(['uncertainty', 'coordination_score'], ascending=[False, False]))
print('Top 10 human-review candidates:')
print(priority_review[review_columns].head(10).to_string(index=False))

Prediction summary by batch:
prediction  Buzzer  Organic
dataset_id                 
batch2         192      815
batch3         248      548
Top 10 human-review candidates:
dataset_id          account_id        username prediction  coordination_score  uncertainty                         text_evidence                                              temporal_evidence                      graph_evidence   review_status
    batch2 2092536425464078336  DaveRemoran1nm    Organic            0.499173     0.499173     nearest=BosPurwa similarity=0.315 1 posts; active span 0.00h; post rate 10.00/h; burst <=1h 0.00   in=0, out=1, total=1, component=2 REVIEW_REQUIRED
    batch3 1552912946569019392 AhmadUl06581878     Buzzer            0.501810     0.498190 nearest=SaptonoRagil similarity=0.419 1 posts; active span 0.00h; post rate 10.00/h; burst <=1h 0.00   in=0, out=1, total=1, component=2 REVIEW_REQUIRED
    batch2 2036700166343069696  ragamceritaidx     Buzzer            0.502128     0.497872    n

In [6]:
artifact_paths = {
    'predictions': ROOT / 'output' / 'phase7' / 'phase7_predictions.csv',
    'priority_review': ROOT / 'output' / 'phase7' / 'phase7_priority_review.csv',
    'score_distribution': ROOT / 'output' / 'phase7' / 'phase7_coordination_score_distribution.png',
    'readiness': ROOT / 'output' / 'phase7' / 'phase7_readiness.json',
    'report': ROOT / 'reports' / 'PHASE7_INFERENCE_EXPLAINABILITY_REPORT.md',
}
missing_outputs = [str(path) for path in artifact_paths.values() if not path.exists()]
assert not missing_outputs, f'Missing Phase 7 output: {missing_outputs}'

notebook_validation = {
    'status': 'PASSED',
    'rows': int(len(predictions)),
    'batches': {batch: int((predictions['dataset_id'] == batch).sum()) for batch in ('batch2', 'batch3')},
    'predicted_buzzer': int((predictions['prediction'] == 'Buzzer').sum()),
    'review_required': int((predictions['review_status'] == 'REVIEW_REQUIRED').sum()),
    'priority_review': int((predictions['review_status'] == 'PRIORITY_REVIEW').sum()),
    'artifacts': {name: str(path) for name, path in artifact_paths.items()},
}
validation_path = ROOT / 'output' / 'phase7' / 'phase7_notebook_validation.json'
validation_path.write_text(json.dumps(notebook_validation, indent=2), encoding='utf-8')
print(json.dumps(notebook_validation, indent=2))
print('Phase 7 notebook validation: PASSED')

{
  "status": "PASSED",
  "rows": 1803,
  "batches": {
    "batch2": 1007,
    "batch3": 796
  },
  "predicted_buzzer": 440,
  "review_required": 291,
  "priority_review": 328,
  "artifacts": {
    "predictions": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase7/phase7_predictions.csv",
    "priority_review": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase7/phase7_priority_review.csv",
    "score_distribution": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase7/phase7_coordination_score_distribution.png",
    "readiness": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase7/phase7_readiness.json",
    "report": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/reports/PHASE7_INFERENCE_EXPLAINABILITY_REPORT.md"
  }
}
Phase 7 notebook validation: PASSED


## Hasil

Output utama tersimpan di `output/phase7/` dan `data/predictions/`.

- `phase7_predictions.csv`: seluruh hasil inference.
- `phase7_priority_review.csv`: kandidat yang perlu ditinjau manusia.
- `phase7_coordination_score_distribution.png`: distribusi score per batch.
- `phase7_notebook_validation.json`: hasil pengecekan notebook.

Status `REVIEW_REQUIRED` dan `PRIORITY_REVIEW` adalah prioritas kerja, bukan ground truth otomatis.